# AAT-3 · RAG over the Neo4j Movies Graph
Nideesh V · 1GA23AI028

Set `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD` (and optionally `LLM_API_KEY`) as environment variables before starting Jupyter. Without `NEO4J_URI` the bundled `data/movies.json` is used.

In [1]:
import os, core
store = core.get_store()
print('Knowledge source:', store.source)
print('Movies:', len(store.movies), '| People:', len(store.people), '| Relationships:', len(store.rels))
print('LLM configured:', bool(os.getenv('LLM_API_KEY')))

Knowledge source: Neo4j database at bolt://localhost:7687
Movies: 38 | People: 133 | Relationships: 250
LLM configured: False


In [2]:
if store.driver:
    rows = store.cypher('MATCH (p:Person)-[r:ACTED_IN]->(m:Movie {title:$t}) RETURN p.name AS actor, r.roles AS roles', t='Top Gun')
    for r in rows: print(r)

{'actor': 'Val Kilmer', 'roles': ['Iceman']}
{'actor': 'Kelly McGillis', 'roles': ['Charlie']}
{'actor': 'Tom Cruise', 'roles': ['Maverick']}
{'actor': 'Anthony Edwards', 'roles': ['Goose']}
{'actor': 'Meg Ryan', 'roles': ['Carole']}
{'actor': 'Tom Skerritt', 'roles': ['Viper']}


In [3]:
found, facts = core.retrieve('Who acted in Top Gun?')
print('Matched:', found)
for f in facts: print(' -', f)

Matched: [('Movie', 'Top Gun')]
 - 'Top Gun' came out in 1986 with the tagline "I feel the need, the need for speed.".
 - The director(s) of 'Top Gun': Tony Scott.
 - The writer(s) of 'Top Gun': Jim Cash.
 - Actors in 'Top Gun': Anthony Edwards (Goose), Kelly McGillis (Charlie), Meg Ryan (Carole), Tom Cruise (Maverick), Tom Skerritt (Viper), Val Kilmer (Iceman).


In [4]:
for q in ['Who acted in Top Gun?', 'What movies did Clint Eastwood direct?', 'Tell me about Jerry Maguire',
          'Which movies were released in 2003?', 'Who wrote A Few Good Men?']:
    r = core.run({'question': q})
    print('Q:', q, '\n  [', r['generator'], ']\n  A:', r['answer'], '\n')

Q: Who acted in Top Gun? 
  [ Template fallback (LLM_API_KEY not set) ]
  A: Based on the Neo4j Movies graph: Actors in 'Top Gun': Anthony Edwards (Goose), Kelly McGillis (Charlie), Meg Ryan (Carole), Tom Cruise (Maverick), Tom Skerritt (Viper), Val Kilmer (Iceman). 

Q: What movies did Clint Eastwood direct? 
  [ Template fallback (LLM_API_KEY not set) ]
  A: Based on the Neo4j Movies graph: Clint Eastwood directed: Unforgiven (1992). 

Q: Tell me about Jerry Maguire 
  [ Template fallback (LLM_API_KEY not set) ]
  A: Based on the Neo4j Movies graph: 'Jerry Maguire' came out in 2000 with the tagline "The rest of his life begins now.". The director(s) of 'Jerry Maguire': Cameron Crowe. The writer(s) of 'Jerry Maguire': Cameron Crowe. 

Q: Which movies were released in 2003? 
  [ Template fallback (LLM_API_KEY not set) ]
  A: Based on the Neo4j Movies graph: Movies from 2003 in the graph: Something's Gotta Give, The Matrix Reloaded, The Matrix Revolutions. 

Q: Who wrote A Few Good Men?

In [5]:
print(core.run({'question': 'Who won the 2011 cricket world cup?'})['answer'])  # nothing relevant in the graph

Based on the Neo4j Movies graph: Movies from 2011 in the graph: none.
